In [15]:
!pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -------- ------------------------------- 1.8/8.3 MB 11.3 MB/s eta 0:00:01
   ---------------------- ----------------- 4.7/8.3 MB 12.6 MB/s eta 0:00:01
   ------------------------------------- -- 7.9/8.3 MB 13.6 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 12.8 MB/s  0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   - -------------------------------------- 1.6/37.3 MB 8.5 MB/s eta 0:00:05
   ---- ----------------------------------- 4.5/37.3 MB 11.9 MB/s eta 0:00:03
   ------- -------------------------------- 7.1/37.3 MB 11.6 MB/s eta 0:00:03
   --------- ------------------------------ 8.9/37.3 MB 11.0 MB/s eta 0:00:03
   ------------ --------------------------- 11.3/37.3 MB 10.7


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
train_data = pd.read_csv("train.csv")


In [10]:
#filling the NA age values with median ages
median_age = train_data['Age'].median()

train_data['Age'].fillna(median_age,inplace = True)
print(train_data.info())

#After doing above steps only 2 NA values were missing which were port they boarded from and hence i will fill it by the most common port that is the Mode of ports
most_common_port = train_data['Embarked'].mode()[0]
train_data['Embarked'].fillna(most_common_port,inplace = True)
print(train_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    

C:\Users\Vansh\AppData\Local\Temp\ipykernel_1320\4049432812.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  train_data['Age'].fillna(median_age,inplace = True)
C:\Users\Vansh\AppData\Local\Temp\ipykernel_1320\4049432812.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignmen

In [11]:
#1 Convert those columns with text into numbers for example 'Sex' to 0s and 1s
train_data['Sex'] = train_data['Sex'].map({'male':0, 'female':1})
#2 INSTEAD OF PORT NAMES LIKE 'S' 'Q' 'C' WE DELETED EMBARKED COLUMN AND ADDED THREE COLUMNS SO IF SOMEONE EMBARKED FROM 'S' THEN 1 IF NOT THEN 0 ETC FOR OTHER PORTS
train_data = pd.get_dummies(train_data,columns=['Embarked'],dtype=int)
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,0,0,1
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,1,0,0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0,0,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,0,0,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,0,0,1


In [31]:
from sklearn.model_selection import train_test_split

#1 Dropping columns which will not play role in making model learn
X = train_data.drop(columns=['Survived','PassengerId','Name','Ticket'])

#2 Seperate your target variables
y = train_data['Survived']

#3 Split data into 80:20 train test split
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.2,random_state=42)

#checking shape to make sure it split correctly
print("Training test shape:", X_train.shape)
print("Validation set shape:", X_val.shape)

Training test shape: (712, 9)
Validation set shape: (179, 9)


In [34]:
from sklearn.metrics import classification_report

#1 Initialize the random forest model
model = RandomForestClassifier(random_state=42)

#2 Train the model
model.fit(X_train, y_train)

#3 predict the surviving chances
predictions = model.predict(X_val)

#4 calculating and printing final result
accuracy = accuracy_score(y_val,predictions)
print(f"Validation Accuracy: {accuracy*100:.2f}%")

Validation Accuracy: 81.01%


In [35]:
# Extract feature importances and pair them with column names
importance = model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance (%)': importance * 100
}).sort_values(by='Importance (%)', ascending=False)

print(feature_importance_df)

      Feature  Importance (%)
1         Sex       27.954744
5        Fare       25.465628
2         Age       25.100399
0      Pclass        9.564949
3       SibSp        4.676460
4       Parch        3.550276
8  Embarked_S        1.551350
6  Embarked_C        1.358438
7  Embarked_Q        0.777756
